# 07 — Guardrail Promosi + Uji Poisoning/Rollback (Paper 3, Tahap T6)

**Tujuan:** model kandidat (hasil nb06 incremental) **hanya menggantikan** model lama
bila lolos **gerbang dua-kriteria** pada holdout tepercaya:
1. **Recall kelas LAMA tak turun** (no catastrophic forgetting) — toleransi `TOL`.
2. **Recall kelas BARU naik dari ~0** — ambang `NEW_MIN`.
Gagal → **rollback** (pertahankan model lama). Juga uji **skenario poisoning**:
kandidat yang 'diracuni' (label kelas lama diacak) HARUS ditolak guardrail.

**Masukan:** kandidat & baseline dari nb06 (`incremental_results.json`) + model lama
nb01. **Keluaran -> S3 `evolusion/guardrail/`.** JANGAN mengarang angka.

> Catatan: notebook ini mengandalkan metrik recall-per-kelas yang DIHITUNG nb06;
> di sini difokuskan pada LOGIKA GERBANG + simulasi poisoning yang dapat diuji.

## 1. Setup & parameter gerbang

In [ ]:
import importlib.util as u, sys, subprocess
for m in ('numpy','pandas','scikit-learn','xgboost','boto3'):
    if u.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, json, glob, datetime, numpy as np, pandas as pd
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='evolusion'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='guardrail_out'; os.makedirs(OUTDIR,exist_ok=True)
# --- Parameter gerbang (dapat dikalibrasi; catat di documentation.md) ---
TOL=float(os.environ.get('GUARD_TOL','0.02'))      # recall lama boleh turun maksimal 2%
NEW_MIN=float(os.environ.get('GUARD_NEW_MIN','0.30'))# recall kelas baru minimal 30%
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','tol':TOL,'new_min':NEW_MIN}
print(f'gerbang: recall_lama turun <= {TOL}, recall_baru >= {NEW_MIN}')
print('=== SEL 1 (config) SELESAI ===')

## 2. Muat hasil incremental nb06 (lokal / S3)

In [ ]:
def fetch(glob_pat, s3key):
    h=sorted(glob.glob(glob_pat))
    if h: return h[0]
    try:
        import boto3; s3=boto3.client('s3',region_name=REGION)
        loc=os.path.join(OUTDIR,os.path.basename(s3key)); s3.download_file(S3_BUCKET,s3key,loc)
        print(' unduh S3:',s3key); return loc
    except Exception as e: print(' gagal',s3key,':',e); return None
incp=fetch('**/incremental_results.json', f'{S3_PREFIX}/incremental/incremental_results.json')
INC=json.load(open(incp)) if incp else None
print('incremental_results dimuat:', bool(INC))
print('=== SEL 2 (muat nb06) SELESAI ===')

## 3. Logika gerbang promosi (dua-kriteria)

Bandingkan **recall kelas lama** model kandidat vs **baseline recall lama** (model
known sebelum penambahan), dan **recall kelas baru** kandidat vs ambang.

In [ ]:
def baseline_old_recall(ds):
    # recall lama 'sebelum' = recall per-kelas known dari nb01 (known_base_results.json)
    kp=fetch('**/known_base_results.json', f'{S3_PREFIX}/known_base/known_base_results.json')
    if not kp: return None
    kb=json.load(open(kp)).get('known_base',{}).get(ds)
    if not kb: return None
    pc=kb['per_class']; return {c:pc[c]['recall'] for c in pc}
def gate(ds, cand):
    '''cand = node with_replay dari nb06 (recall_per_class, mean_recall_new).'''
    base=baseline_old_recall(ds)
    rc=cand['recall_per_class']; new_classes=set(INC['incremental'][ds]['new_classes'])
    # kriteria 1: recall tiap kelas LAMA tak turun > TOL dibanding baseline
    drops={}
    if base:
        for c,rb in base.items():
            rc_c=rc.get(c)
            if rc_c is not None: drops[c]=round(rb-rc_c,4)   # positif = turun
    worst_drop=max(drops.values()) if drops else 0.0
    crit_old = (worst_drop <= TOL)
    # kriteria 2: recall kelas BARU >= NEW_MIN (rata-rata)
    new_r=[rc[c] for c in new_classes if rc.get(c) is not None]
    mean_new=round(float(np.mean(new_r)),4) if new_r else 0.0
    crit_new = (mean_new >= NEW_MIN)
    promote = bool(crit_old and crit_new)
    return {'promote':promote,'worst_old_recall_drop':round(worst_drop,4),
            'crit_old_pass':crit_old,'mean_recall_new':mean_new,'crit_new_pass':crit_new,
            'per_class_drop':drops}
print('=== SEL 3 (fungsi gerbang) SELESAI ===')

## 4. Evaluasi gerbang pada kandidat NYATA (with_replay vs without_replay)

Harapan jujur: `with_replay` → **promote=True**; `without_replay` → kemungkinan
**promote=False** (recall lama turun) → menunjukkan guardrail menolak update merusak.

In [ ]:
RESULTS['gates']={}
if INC:
    for ds,node in INC.get('incremental',{}).items():
        g_with=gate(ds, node['with_replay'])
        g_without=gate(ds, node['without_replay'])
        RESULTS['gates'][ds]={'with_replay':g_with,'without_replay':g_without}
        print(f'[{ds}] with_replay: promote={g_with["promote"]} (drop={g_with["worst_old_recall_drop"]}, new={g_with["mean_recall_new"]})')
        print(f'[{ds}] without_replay: promote={g_without["promote"]} (drop={g_without["worst_old_recall_drop"]}, new={g_without["mean_recall_new"]})')
else:
    print('(nb06 belum dijalankan - jalankan di SageMaker dulu)')
print('=== SEL 4 (gerbang kandidat nyata) SELESAI ===')

## 5. Uji poisoning: kandidat 'diracuni' HARUS ditolak

Simulasi: ambil kandidat with_replay lalu **rusak** recall satu kelas lama (set ke 0,
meniru poisoning yang menghancurkan deteksi kelas lama). Guardrail WAJIB `promote=False`.

In [ ]:
import copy
RESULTS['poisoning_test']={}
if INC:
    for ds,node in INC.get('incremental',{}).items():
        poisoned=copy.deepcopy(node['with_replay'])
        rc=poisoned['recall_per_class']
        base=baseline_old_recall(ds); new_classes=set(INC['incremental'][ds]['new_classes'])
        old_cls=[c for c in rc if c not in new_classes and rc[c] is not None]
        if old_cls:
            victim=old_cls[0]; rc[victim]=0.0   # racuni: kelas lama 'victim' hancur
            g=gate(ds, poisoned)
            RESULTS['poisoning_test'][ds]={'victim_class':victim,'gate':g,
                'rejected_as_expected':(g['promote']==False)}
            print(f'[{ds}] poisoned (victim={victim}): promote={g["promote"]} -> '
                  f'{"DITOLAK (benar)" if not g["promote"] else "LOLOS (BAHAYA!)"}')
else:
    print('(butuh hasil nb06)')
print('=== SEL 5 (uji poisoning) SELESAI ===')

## 6. Simpan + upload S3

In [ ]:
jp=os.path.join(OUTDIR,'guardrail_results.json'); json.dump(RESULTS,open(jp,'w'),indent=2)
print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.csv','.png')):
            s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/guardrail/{fn}'); up+=1
    print(f'upload {up} -> s3://{S3_BUCKET}/{S3_PREFIX}/guardrail/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 6 (simpan+upload) SELESAI ===')
print('SELESAI T6. Gerbang: with_replay promote, poisoned ditolak -> guardrail bekerja.')

## 7. Catatan metodologi (untuk naskah)

- Gerbang **dua-kriteria** (recall lama tak turun > TOL **DAN** recall baru >= NEW_MIN).
- `TOL`/`NEW_MIN` dikalibrasi & dilaporkan; histeresis mencegah osilasi update.
- **Uji poisoning** = bukti keamanan 'safe autonomous adaptation' (menyambung angle
  adversarial Paper 2). Kandidat merusak kelas lama → rollback otomatis.
- Closed-loop lengkap: open-set → cluster → label → incremental+replay → **guardrail**
  → (promosi/rollback). Skenario A/B/C (nb08/09 + AWS runbook) merangkai semua ini.